In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))

import numpy as np
import pandas as pd
from src.utils.pipeline import load_all_snapshots
from src.data.validation import plate_appearances

SEASONS = [2021, 2022, 2023, 2024]
df = load_all_snapshots(seasons=SEASONS)
print(df.shape)

pa = plate_appearances(df)
pa["season"] = pd.to_datetime(pa["game_date"]).dt.year

# Batter season lines. Intentional walks are excluded from BB% (Day 3):
# they reflect the manager's decision, not the hitter's plate skill.
lines = (
    pa.groupby(["batter", "season"])["events"]
    .agg(
        pa_count="size",
        k=lambda s: (s == "strikeout").sum(),
        bb=lambda s: (s == "walk").sum(),
    )
    .reset_index()
)
lines["k_pct"] = lines["k"] / lines["pa_count"]
lines["bb_pct"] = lines["bb"] / lines["pa_count"]

print(lines.groupby("season").agg(
    players=("batter", "size"),
    league_k=("k", "sum"),
    league_pa=("pa_count", "sum"),
).assign(k_pct=lambda d: d["league_k"] / d["league_pa"]).round(4).to_string())

(2849203, 119)
        players  league_k  league_pa   k_pct
season                                      
2021       1047     41773     180864  0.2310
2022        693     40691     182349  0.2231
2023        656     41743     184376  0.2264
2024        651     41020     182436  0.2248


In [2]:
MARCEL_WEIGHTS = [5, 4, 3]      # most recent first, verified: batting
MARCEL_REGRESSION_PA = 100      # verified: batting

def marcel_projection(lines, target_season, metric, weights=MARCEL_WEIGHTS,
                      regression_pa=MARCEL_REGRESSION_PA):
    """Marcel for one metric and one projection season.

    Weighted three-year rate, regressed toward the league average with
    `regression_pa` plate appearances of league-average performance.

    NO age adjustment. See the docs for why.
    """
    prior_seasons = [target_season - i for i in (1, 2, 3)]

    num = pd.Series(0.0)
    den = pd.Series(0.0)
    frames = []
    for w, s in zip(weights, prior_seasons):
        sub = lines[lines["season"] == s].set_index("batter")
        frames.append(sub[[metric.replace("_pct", ""), "pa_count"]].mul(w))

    combined = pd.concat(frames, axis=1)
    stat_cols = [c for c in combined.columns if c != "pa_count"]
    weighted_stat = combined[stat_cols].sum(axis=1, min_count=1)
    weighted_pa = combined["pa_count"].sum(axis=1, min_count=1)

    # League rate over the same weighted window
    league_rate = weighted_stat.sum() / weighted_pa.sum()

    projected = ((weighted_stat + league_rate * regression_pa * sum(weights))
                 / (weighted_pa + regression_pa * sum(weights)))

    out = pd.DataFrame({
        f"marcel_{metric}": projected,
        "weighted_pa": weighted_pa / sum(weights),
        "league_rate": league_rate,
    })
    return out.dropna(subset=[f"marcel_{metric}"])

proj_k = marcel_projection(lines, 2024, "k_pct")
proj_bb = marcel_projection(lines, 2024, "bb_pct")

print(f"K%  projections: {len(proj_k)}")
print(proj_k.describe().round(4).to_string())

K%  projections: 1337
       marcel_k_pct  weighted_pa  league_rate
count     1337.0000    1337.0000    1337.0000
mean         0.6967     136.7407       0.6793
std          0.0884     182.9877       0.0000
min          0.2799       0.2500       0.6793
25%          0.6742       2.7500       0.6793
50%          0.6909      38.5000       0.6793
75%          0.7374     223.2500       0.6793
max          1.0728     736.4167       0.6793


In [3]:
actual = lines[lines["season"] == 2024].set_index("batter")

MIN_PA = 200
evalset = proj_k.join(actual[["k_pct", "pa_count"]], rsuffix="_actual")
evalset = evalset[(evalset["pa_count"] >= MIN_PA) & (evalset["weighted_pa"] >= 100)]

print(f"{len(evalset)} batters with 200+ PA in 2024 and prior history")
print()

def evaluate(pred, actual_vals, name):
    return {
        "model": name,
        "mae": np.abs(pred - actual_vals).mean(),
        "rmse": np.sqrt(((pred - actual_vals) ** 2).mean()),
        "corr": np.corrcoef(pred, actual_vals)[0, 1],
    }

results = [
    evaluate(evalset["marcel_k_pct"], evalset["k_pct"], "Marcel"),
    evaluate(pd.Series(evalset["league_rate"].iloc[0], index=evalset.index),
             evalset["k_pct"], "league average"),
]
print(pd.DataFrame(results).round(4).to_string(index=False))

298 batters with 200+ PA in 2024 and prior history

         model    mae   rmse   corr
        Marcel 0.4447 0.4534 0.8176
league average 0.4606 0.4644 0.0000


In [4]:
print(lines[lines["season"] == 2021]["pa_count"].describe().round(1).to_string())
print()
print(lines[lines["season"] == 2022]["pa_count"].describe().round(1).to_string())

count    1047.0
mean      172.7
std       207.2
min         1.0
25%         5.5
50%        62.0
75%       298.0
max       721.0

count    693.0
mean     263.1
std      213.2
min        1.0
25%       62.0
50%      220.0
75%      441.0
max      726.0


In [5]:
def marcel_projection(lines, target_season, stat_col, weights=(5, 4, 3),
                      regression_pa=100):
    """Marcel: weighted 3-year rate regressed toward the league average.

    stat_col is the COUNT column ("k" or "bb"), not the rate.
    No age adjustment — see docs.
    """
    prior = [target_season - i for i in (1, 2, 3)]

    num = None
    den = None
    for w, season in zip(weights, prior):
        sub = lines[lines["season"] == season].set_index("batter")
        n = sub[stat_col] * w
        d = sub["pa_count"] * w
        num = n if num is None else num.add(n, fill_value=0)
        den = d if den is None else den.add(d, fill_value=0)

    league_rate = num.sum() / den.sum()
    reg = regression_pa * sum(weights)

    projected = (num + league_rate * reg) / (den + reg)

    return pd.DataFrame({
        "projection": projected,
        "weighted_pa": den / sum(weights),
        "league_rate": league_rate,
    })

proj_k = marcel_projection(lines, 2024, "k")
print("league rate:", round(proj_k["league_rate"].iloc[0], 4), "(expect ~0.225)")
print(proj_k["projection"].describe().round(4).to_string())

league rate: 0.2264 (expect ~0.225)
count    1337.0000
mean        0.2322
std         0.0295
min         0.0933
25%         0.2247
50%         0.2303
75%         0.2458
max         0.3576


In [6]:
# Pitchers batted in the NL through 2021; the universal DH began in 2022.
# Including their plate appearances pollutes both the league rate and the
# player pool — 1,047 "batters" in 2021 versus ~660 after.
pitcher_ids = set(df["pitcher"].dropna().unique())
lines_bat = lines[~lines["batter"].isin(pitcher_ids)].copy()

print("before:", lines.groupby("season").size().to_dict())
print("after: ", lines_bat.groupby("season").size().to_dict())

before: {2021: 1047, 2022: 693, 2023: 656, 2024: 651}
after:  {2021: 513, 2022: 542, 2023: 523, 2024: 548}


In [7]:
proj_k = marcel_projection(lines_bat, 2024, "k")
proj_bb = marcel_projection(lines_bat, 2024, "bb")

print("K%  league rate:", round(proj_k["league_rate"].iloc[0], 4))
print("BB% league rate:", round(proj_bb["league_rate"].iloc[0], 4))
print()
print(proj_k["projection"].describe().round(4).to_string())

K%  league rate: 0.2237
BB% league rate: 0.0835

count    767.0000
mean       0.2305
std        0.0342
min        0.0929
25%        0.2156
50%        0.2308
75%        0.2480
max        0.3571


In [8]:
actual = lines_bat[lines_bat["season"] == 2024].set_index("batter")

def build_eval(proj, actual, metric_col, min_pa=200, min_history=100):
    e = proj.join(actual[[metric_col, "pa_count"]], how="inner")
    e["actual_rate"] = e[metric_col] / e["pa_count"]
    return e[(e["pa_count"] >= min_pa) & (e["weighted_pa"] >= min_history)]

ek = build_eval(proj_k, actual, "k")
ebb = build_eval(proj_bb, actual, "bb")

def evaluate(pred, truth, name):
    return {"model": name,
            "mae": np.abs(pred - truth).mean(),
            "rmse": np.sqrt(((pred - truth) ** 2).mean()),
            "corr": np.corrcoef(pred, truth)[0, 1]}

for name, e in [("K%", ek), ("BB%", ebb)]:
    print(f"=== {name}  ({len(e)} batters) ===")
    rows = [
        evaluate(e["projection"], e["actual_rate"], "Marcel"),
        evaluate(pd.Series(e["league_rate"].iloc[0], index=e.index),
                 e["actual_rate"], "league average"),
    ]
    print(pd.DataFrame(rows).round(4).to_string(index=False))
    print()

=== K%  (257 batters) ===
         model    mae   rmse   corr
        Marcel 0.0271 0.0348 0.8078
league average 0.0476 0.0587 0.0000

=== BB%  (257 batters) ===
         model    mae   rmse   corr
        Marcel 0.0143 0.0179 0.7627
league average 0.0220 0.0272    NaN



/Users/minjong/Projects/mlb-intelligence-lab/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/minjong/Projects/mlb-intelligence-lab/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [9]:
# "Last season only" — the naive projection everyone reaches for first.
prev = lines_bat[lines_bat["season"] == 2023].set_index("batter")
for name, e, col in [("K%", ek, "k"), ("BB%", ebb, "bb")]:
    naive = (prev[col] / prev["pa_count"]).reindex(e.index)
    mask = naive.notna()
    print(f"{name} last-season-only:",
          evaluate(naive[mask], e.loc[mask, "actual_rate"], "prev season"))

K% last-season-only: {'model': 'prev season', 'mae': np.float64(0.028651703561697016), 'rmse': np.float64(0.03898951908313737), 'corr': np.float64(0.7853038470855935)}
BB% last-season-only: {'model': 'prev season', 'mae': np.float64(0.01692940037346064), 'rmse': np.float64(0.0221159231123094), 'corr': np.float64(0.7230296172052484)}
